**11주차 복습과제**

📍Distributed representations of words and phrases and their compositionality

교재 :  <(파이토치 트랜스포머를 활용한) 자연어 처리와 컴퓨터 비전 심층학습>

범위 :  p.286~p.298 Word2Vec 코드 필사

###모델 실습: Skip-gram

In [ ]:
#임베딩 클래스
embedding = torch.nn.Embedding(
    num_embeddings,
    embedding_dim,
    padding_idx=None,
    max_norm=None,
    norm_type=2.0
)

In [8]:
#기본 Skip-gram 클래스
from torch import nn

class VanillaSkipgram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim
        )
        self.linear = nn.Linear(
            in_features=embedding_dim,
            out_features=vocab_size
        )

    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        output = self.linear(embeddings)
        return output

In [1]:
!pip install Korpora
!pip install konlpy

  Using cached konlpy-0.6.0-py2.py3-none-any.whl.metadata (1.9 kB)
  Using cached jpype1-1.7.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (4.3 kB)
Using cached konlpy-0.6.0-py2.py3-none-any.whl (19.4 MB)
Using cached jpype1-1.7.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (438 kB)


In [2]:
#영화 리뷰 데이터세트 전처리
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)

tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 236MB/s]
[nsmc] download ratings_test.txt: 4.90MB [00:00, 142MB/s]


[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


In [3]:
#단어 사전 구축
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

vocab = build_vocab(corpus=tokens, n_vocab=5000, special_tokens=["<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [4]:
#Skip-gram의 단어 쌍 추출
def get_word_pairs(tokens, window_size):
    pairs = []
    for sentence in tokens:
        sentence_length = len(sentence)
        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx - window_size)
            window_end = min(sentence_length, idx + window_size + 1)
            center_word = sentence[idx]
            context_words = sentence[window_start:idx] + sentence[idx+1:window_end]
            for context_word in context_words:
                pairs.append([center_word, context_word])
    return pairs

word_pairs = get_word_pairs(tokens, window_size=2)
print(word_pairs[:5])

[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


In [5]:
#인덱스 쌍 변환
def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id["<unk>"]
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs

index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]


In [6]:
#데이터로더 적용
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs = torch.tensor(index_pairs)
center_indexs = index_pairs[:, 0]
contenxt_indexs = index_pairs[:, 1]

dataset = TensorDataset(center_indexs, contenxt_indexs)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [9]:
#Skip-gram 모델 준비 작업
from torch import optim

device = "cuda" if torch.cuda.is_available() else "cpu"
word2vec = VanillaSkipgram(vocab_size=len(token_to_id), embedding_dim=128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr=0.1)

In [10]:
#모델 학습
for epoch in range(10):
    cost = 0.0

    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss

    cost = cost / len(dataloader)
    print(f"Epoch : {epoch+1:4d}, Cost : {cost:.3f}")

Epoch :    1, Cost : 6.196
Epoch :    2, Cost : 5.981
Epoch :    3, Cost : 5.931
Epoch :    4, Cost : 5.900
Epoch :    5, Cost : 5.878
Epoch :    6, Cost : 5.861
Epoch :    7, Cost : 5.846
Epoch :    8, Cost : 5.833
Epoch :    9, Cost : 5.821
Epoch :   10, Cost : 5.811


In [11]:
#임베딩 값 추출
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()

for word, embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding

index = 30
token = vocab[30]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)

연기
[-0.18400604  0.62585855  2.5870113   1.8202302  -2.125531   -1.4719528
  0.22089218 -0.67745835  0.599733    0.2408836   0.9791346   0.22220396
  0.99404025  0.5435096  -1.0434078  -0.4624183   0.00583886 -0.10433228
  0.06372139 -0.05278103  1.3537724   1.2041363   1.4195092  -0.3965381
 -1.0995232  -0.06131869 -0.78401697  0.8611713   1.550264    1.1031896
  0.3593944   0.8890287  -0.30359316 -1.5327932  -0.54297584 -0.09942979
 -1.4097918  -0.70168585  0.5238408   0.9478174   0.18658543 -0.5255445
 -0.8725158  -0.9100745   0.34982374 -0.65098685 -0.17680787  1.143673
  0.8889106  -0.31156093  0.2832295  -0.3992775   2.3293138  -0.03693467
  0.37927473  0.1311174  -0.8027599   1.1031458  -0.78036726  0.42254987
 -0.36949763 -0.4893204  -1.5294338   0.11366042 -2.023581    0.93191683
  1.3417741  -0.06778293 -0.27957618  1.3161583  -0.28301573  1.089561
 -0.73898965 -0.21385852 -1.0243958   1.3711745   0.23522863 -0.8254389
 -0.42927858  0.04157196 -1.1387606   0.9499263  -0.04682

In [12]:
#단어 임베딩 유사도 계산
import numpy as np
from numpy.linalg import norm

def cosine_similarity(a, b):
    cosine = np.dot(b, a)/(norm(b, axis=1)*norm(a))
    return cosine

def top_n_index(cosine_matrix, n):
    closest_indexes = cosine_matrix.argsort()[::-1]
    top_n = closest_indexes[1 : n + 1]
    return top_n

cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n=5)

print(f"{token}와 가장 유사한 5 개 단어")
for index in top_n:
    print(f"{id_to_token[index]} - 유사도: {cosine_matrix[index]:.4f}")

연기와 가장 유사한 5 개 단어
상관 - 유사도: 0.2918
기술 - 유사도: 0.2913
대사 - 유사도: 0.2911
좋은데 - 유사도: 0.2779
유 - 유사도: 0.2622
